# **DATA PROFILING**

Import libraries:

In [11]:
import sys
!{sys.executable} -m pip install -U ydata-profiling[notebook]
!pip install jupyter-contrib-nbextensions

  Using cached jupyter_contrib_nbextensions-0.7.0-py2.py3-none-any.whl
  Using cached jupyter_contrib_core-0.4.2-py2.py3-none-any.whl
  Using cached jupyter_highlight_selected_word-0.2.0-py2.py3-none-any.whl.metadata (730 bytes)
  Using cached jupyter_nbextensions_configurator-0.6.4-py2.py3-none-any.whl.metadata (1.8 kB)
Using cached jupyter_highlight_selected_word-0.2.0-py2.py3-none-any.whl (11 kB)
Using cached jupyter_nbextensions_configurator-0.6.4-py2.py3-none-any.whl (466 kB)


In [12]:
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [13]:
from ydata_profiling import ProfileReport
from ydata_profiling.config import Settings
import pandas as pd
import numpy as np
import json

Import data:

In [14]:
ARTWORKS = pd.read_csv("https://raw.githubusercontent.com/omar-mohamed-azab/Polimi-Data-and-Information-Quality-Project/refs/heads/main/Dataset/artwork_data.csv")
artists = pd.read_csv("https://raw.githubusercontent.com/omar-mohamed-azab/Polimi-Data-and-Information-Quality-Project/refs/heads/main/Dataset/artist_data.csv")

/tmp/ipython-input-390809704.py:1: DtypeWarning: Columns (9,13) have mixed types. Specify dtype option on import or set low_memory=False.
  ARTWORKS = pd.read_csv("https://raw.githubusercontent.com/omar-mohamed-azab/Polimi-Data-and-Information-Quality-Project/refs/heads/main/Dataset/artwork_data.csv")


In [15]:
ARTWORKS.columns

Index(['id', 'accession_number', 'artist', 'artistRole', 'artistId', 'title',
       'dateText', 'medium', 'creditLine', 'year', 'acquisitionYear',
       'dimensions', 'width', 'height', 'depth', 'units', 'inscription',
       'thumbnailCopyright', 'thumbnailUrl', 'url'],
      dtype='object')

In [16]:
ARTWORKS.shape

(69201, 20)

In [17]:
ARTWORKS.head()

,id,accession_number,artist,artistRole,artistId,title,dateText,medium,creditLine,year,acquisitionYear,dimensions,width,height,depth,units,inscription,thumbnailCopyright,thumbnailUrl,url
0,1035,A00001,"Blake, Robert",artist,38,A Figure Bowing before a Seated Old Man with h...,date not known,"Watercolour, ink, chalk and graphite on paper....",Presented by Mrs John Richmond 1922,NaN,1922.0,support: 394 x 419 mm,394,419.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-a-fi...
1,1036,A00002,"Blake, Robert",artist,38,"Two Drawings of Frightened Figures, Probably f...",date not known,Graphite on paper,Presented by Mrs John Richmond 1922,NaN,1922.0,support: 311 x 213 mm,311,213.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-two-...
2,1037,A00003,"Blake, Robert",artist,38,The Preaching of Warning. Verso: An Old Man En...,?c.1785,Graphite on paper. Verso: graphite on paper,Presented by Mrs John Richmond 1922,1785.0,1922.0,support: 343 x 467 mm,343,467.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...
3,1038,A00004,"Blake, Robert",artist,38,Six Drawings of Figures with Outstretched Arms,date not known,Graphite on paper,Presented by Mrs John Richmond 1922,NaN,1922.0,support: 318 x 394 mm,318,394.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-six-...
4,1039,A00005,"Blake, William",artist,39,The Circle of the Lustful: Francesca da Rimini...,"1826–7, reprinted 1892",Line engraving on paper,Purchased with the assistance of a special gra...,1826.0,1919.0,image: 243 x 335 mm,243,335.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...


In [18]:
ARTWORKS.dtypes

,0
id,int64
accession_number,object
artist,object
artistRole,object
artistId,int64
title,object
dateText,object
medium,object
creditLine,object
year,object


In [19]:
config=Settings()
config.vars.text.words = False

profile = ProfileReport(ARTWORKS, title = "ARTWORKS_REPORT", config = config)

In [20]:
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 20/20 [00:17<00:00,  1.15it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

# **DATA CLEANING**

*DATA TRANSFORMATION/NORMALIZATION*

- Merging height and width into dimensions if they exist
- Uossibly removing depth (96.4% missing).
- Units has no variance, remove
- Inscription also no variance but a lot of missing values, possibly changing to dateInscription with True/False if thats the semantic, otherwise removing.
- Thumbnail copyright and thumbnail url needed? Some missing values. Picture available through the “normal” url column with 0 missing values.
- Artistrole mostly contains “Artist”, highly imbalanced.
- Possibly removing one of id and accession_number since both are unique (also Url but might be harder as an identifier). Id weirdly distributed as an argument för using accession_number as id instead.
- Year type and format needs to be addressed.
- Using artist_data for missing year, perhaps adding uncertainty column
- Making sure every artist_id, name pair is unique.
- Titles not unique, issue? Maybe not.
- Date format.
- Medium: handle missing values.


In [51]:
#Set accession_number as index
df = ARTWORKS.set_index('accession_number')
df

,id,artist,artistRole,artistId,title,dateText,medium,creditLine,year,acquisitionYear,dimensions,width,height,depth,units,inscription,thumbnailCopyright,thumbnailUrl,url
accession_number,,,,,,,,,,,,,,,,,,,
A00001,1035,"Blake, Robert",artist,38,A Figure Bowing before a Seated Old Man with h...,date not known,"Watercolour, ink, chalk and graphite on paper....",Presented by Mrs John Richmond 1922,NaN,1922.0,support: 394 x 419 mm,394,419.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-a-fi...
A00002,1036,"Blake, Robert",artist,38,"Two Drawings of Frightened Figures, Probably f...",date not known,Graphite on paper,Presented by Mrs John Richmond 1922,NaN,1922.0,support: 311 x 213 mm,311,213.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-two-...
A00003,1037,"Blake, Robert",artist,38,The Preaching of Warning. Verso: An Old Man En...,?c.1785,Graphite on paper. Verso: graphite on paper,Presented by Mrs John Richmond 1922,1785.0,1922.0,support: 343 x 467 mm,343,467.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...
A00004,1038,"Blake, Robert",artist,38,Six Drawings of Figures with Outstretched Arms,date not known,Graphite on paper,Presented by Mrs John Richmond 1922,NaN,1922.0,support: 318 x 394 mm,318,394.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-six-...
A00005,1039,"Blake, William",artist,39,The Circle of the Lustful: Francesca da Rimini...,"1826–7, reprinted 1892",Line engraving on paper,Purchased with the assistance of a special gra...,1826.0,1919.0,image: 243 x 335 mm,243,335.0,NaN,mm,NaN,NaN,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T13865,122960,"P-Orridge, Genesis",artist,16646,Larvae (from Tampax Romana),1975,"Perspex, Wood, hairpiece, tampon and human blood",Transferred from Tate Archive 2012,1975,2013.0,object: 305 x 305 x 135 mm,305,305.0,135.0,mm,NaN,NaN,NaN,http://www.tate.org.uk/art/artworks/p-orridge-...
T13866,122961,"P-Orridge, Genesis",artist,16646,Living Womb (from Tampax Romana),1976,"Wood, Perspex, plastic, photograph on paper, t...",Transferred from Tate Archive 2012,1976,2013.0,object: 305 x 305 x 135 mm,305,305.0,135.0,mm,NaN,NaN,NaN,http://www.tate.org.uk/art/artworks/p-orridge-...
T13867,121181,"Hatoum, Mona",artist,2365,Present Tense,1996,Soap and glass beads,Presented by Tate Members 2013,1996,2013.0,displayed: 45 x 2410 x 2990 mm,45,2410.0,2990.0,mm,NaN,NaN,NaN,http://www.tate.org.uk/art/artworks/hatoum-pre...


In [52]:
# transform numeric columns
numeric_cols = ['width', 'height', 'depth', 'year']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [53]:
# Create function to merge width, height and depth into a single column if existent
def merge_dimensions(row):
    # If height or width is missing, we can't create a valid dimension
    if pd.isna(row['height']) or pd.isna(row['width']):
        return "Dimensions not known"

    # Base: Height x Width
    dim_str = f"{row['height']} x {row['width']}"

    # Add Depth only if it exists and is not 0
    if pd.notna(row['depth']) and row['depth'] > 0:
        dim_str += f" x {row['depth']}"

    return dim_str + " mm"

df['dimensions'] = df.apply(merge_dimensions, axis=1)

# Show some examples with depth and some without
print(df[['dimensions']].tail(10))

                                 dimensions
accession_number                           
T13860                 Dimensions not known
T13861                    1155.0 x 810.0 mm
T13862                 Dimensions not known
T13863             305.0 x 305.0 x 135.0 mm
T13864             305.0 x 305.0 x 135.0 mm
T13865             305.0 x 305.0 x 135.0 mm
T13866             305.0 x 305.0 x 135.0 mm
T13867            2410.0 x 45.0 x 2990.0 mm
T13868                 Dimensions not known
T13869                     660.0 x 508.0 mm


In [54]:
# We want to fill empty cells of 'year' with an approximation, while also representing the uncertainty in a separate variable

print("Artist Columns:", artists.columns.tolist())

# Select the correct columns (renaming them to be consistent with our logic)
artist_dates = artists[['id', 'yearOfBirth', 'yearOfDeath']].rename(
    columns={'yearOfBirth': 'birthYear', 'yearOfDeath': 'deathYear'}
)

# 2. RE-MERGE
# Merge them into the main dataframe
df = df.reset_index().merge(
    artist_dates,
    left_on='artistId',
    right_on='id',
    how='left',
    suffixes=('', '_artist_numeric')
).set_index('accession_number')

df = df.drop(columns=['id_artist_numeric'], errors='ignore')

# Define approximation logic
def estimate_year_logic(row):
    # Case A: We already have a valid year -> "Verified"
    if pd.notna(row['year']):
        return pd.Series([row['year'], "Verified", f"{int(row['year'])}"])

    # Case B: Missing year, but we have Artist Birth/Death
    if pd.notna(row['birthYear']) and pd.notna(row['deathYear']):
        # Assumption: Artist starts working at age 20
        start_career = row['birthYear'] + 20
        end_career = row['deathYear']

        # Check validity (sometimes death is missing or invalid)
        if end_career > start_career:
            mid_point = start_career + (end_career - start_career) / 2
            label = f"c. {int(start_career)}–{int(end_career)} (Artist Era)"
            return pd.Series([int(mid_point), "Estimated", label])

    # Case C: No data at all -> Remain Missing
    return pd.Series([np.nan, "Unknown", "Date not known"])

print("Calculating estimates for missing years...")
df[['year_numeric', 'year_certainty', 'date_display']] = df.apply(estimate_year_logic, axis=1)

estimated_mask = df['year_certainty'] == 'Estimated'

print(df[estimated_mask][['artist','year_numeric', 'year_certainty', 'date_display']].sample(10))

Artist Columns: ['id', 'name', 'gender', 'dates', 'yearOfBirth', 'yearOfDeath', 'placeOfBirth', 'placeOfDeath', 'url']
Calculating estimates for missing years...
                                          artist  year_numeric year_certainty  \
accession_number                                                                
A01378                             Jones, George        1837.0      Estimated   
D36671            Turner, Joseph Mallord William        1823.0      Estimated   
D36664            Turner, Joseph Mallord William        1823.0      Estimated   
T08169                               Kinnard, W.        1827.0      Estimated   
T08582                              Barry, James        1783.0      Estimated   
N02047                           Stevens, Alfred        1856.0      Estimated   
A01601                             Jones, George        1837.0      Estimated   
A01579                             Jones, George        1837.0      Estimated   
A00549                      

In [55]:
top_titles = df['title'].value_counts().head(20)
print(top_titles)

title
[title not known]                                3572
Blank                                            3031
[blank]                                          2482
[no title]                                       1874
Untitled                                          639
Mountains                                         538
[inscriptions by Turner]                          209
Shipping                                          202
Buildings                                         156
River Scene                                       109
Study of Sky                                      101
Printed Page of Coltman’s ‘British Itinerary’      92
Inscription by Turner: Draft of Poetry             81
Coast Scene                                        73
Cliffs                                             69
Landscape                                          67
Castle on Rock                                     67
[inscription by Turner]                            66
Sketches              

In [56]:
# clean 'Untitled' variations and inscriptions
untitled_variants = [
    '[title not known]',
    'Blank',
    '[blank]',
    '[no title]'
]

df['title_cleaned'] = df['title'].replace(untitled_variants, 'Untitled')

mask_inscription = df['title_cleaned'].str.contains(r'inscription', case=False, regex=True)
df.loc[mask_inscription, 'title_cleaned'] = 'Inscription (Various)'
print(df['title_cleaned'].value_counts().head(10))

title_cleaned
Untitled                                         11598
Inscription (Various)                             1268
Mountains                                          538
Shipping                                           202
Buildings                                          156
River Scene                                        109
Study of Sky                                       101
Printed Page of Coltman’s ‘British Itinerary’       92
Coast Scene                                         73
Cliffs                                              69
Name: count, dtype: int64


In [57]:
# check for name variations for artistID
name_counts = df.groupby('artistId')['artist'].nunique()
inconsistent_ids = name_counts[name_counts > 1]
inconsistent_ids

,artist
artistId,


In [58]:
# Check for inconsistencies between names and id pairs in artworks and artist dataset
official_names = artists.set_index('id')['name']
mask_mismatch = (
    df['artistId'].isin(official_names.index) &
    (df['artist'].str.strip() != df['artistId'].map(official_names).str.strip())
)

mismatches = df[mask_mismatch]
mismatches

,id,artist,artistRole,artistId,title,dateText,medium,creditLine,year,acquisitionYear,...,inscription,thumbnailCopyright,thumbnailUrl,url,birthYear,deathYear,year_numeric,year_certainty,date_display,title_cleaned
accession_number,,,,,,,,,,,,,,,,,,,,,


In [59]:
# Check medium variation
top_mediums = df['medium'].value_counts().head(50)
print(top_mediums)

medium
Graphite on paper                                  26167
Oil paint on canvas                                 3383
Screenprint on paper                                2984
Lithograph on paper                                 2721
Watercolour on paper                                1890
Etching on paper                                    1793
Graphite and watercolour on paper                   1680
Ink on paper                                         880
Intaglio print on paper                              820
Photograph, gelatin silver print on paper            750
Engraving on paper                                   732
Line engraving on paper                              645
Chalk and graphite on paper                          619
Pen and ink on paper                                 598
Chalk on paper                                       526
Gouache and watercolour on paper                     475
Print on paper                                       435
Aquatint on paper       

In [60]:
# Make new boolean column 'has_inscription' to replace 'inscription'
if 'inscription' in df.columns:
    df['has_inscription'] = df['inscription'].notna()

In [61]:
# Fill empty cells for creditLine, medium and thumbnailCopyright
df['creditLine'] = df['creditLine'].fillna("No credit line recorded")
df['medium'] = df['medium'].fillna("Unknown Medium")
df['thumbnailCopyright'] = df['thumbnailCopyright'].fillna("No copyright information available")
df['thumbnailUrl'] = df['thumbnailUrl'].fillna("No thumbnail URL available")

In [62]:
# Lets group the different mediums into larger groups
def clean_medium(text):
    # Convert to string and lowercase to avoid case issues
    text = str(text).lower()

    # Modern/New Media
    if any(x in text for x in ['video', 'film', 'audio', 'digital', 'screenprint', 'photograph', 'c-print', 'plastic', 'acrylic', 'installation']):
        return 'Modern Media'

    # Sculptures (3D Materials)
    if any(x in text for x in ['bronze', 'marble', 'stone', 'plaster', 'sculpture', 'metal', 'steel', 'wood', 'glass']):
        return 'Sculpture'

    # Paintings (Traditional)
    if any(x in text for x in ['oil', 'canvas', 'tempera', 'panel', 'paint', 'gouache']):
        return 'Painting'

    # Works on Paper (Drawings, Prints)
    if any(x in text for x in ['paper', 'ink', 'graphite', 'watercolour', 'chalk', 'pencil', 'print', 'etching', 'engraving', 'lithograph']):
        return 'Work on Paper'

    # Everything else
    return 'Other/Unknown'

df['medium_group'] = df['medium'].apply(clean_medium)


# Compare the mess of the original vs the clean version
print("--- Before (Top 5 Raw) ---")
print(df['medium'].value_counts().head(5))

print("\n--- After (Clean Categories) ---")
print(df['medium_group'].value_counts())

--- Before (Top 5 Raw) ---
medium
Graphite on paper       26167
Unknown Medium           6384
Oil paint on canvas      3383
Screenprint on paper     2984
Lithograph on paper      2721
Name: count, dtype: int64

--- After (Clean Categories) ---
medium_group
Work on Paper    46438
Other/Unknown     7072
Painting          6603
Modern Media      6510
Sculpture         2578
Name: count, dtype: int64


In [63]:
df.columns

Index(['id', 'artist', 'artistRole', 'artistId', 'title', 'dateText', 'medium',
       'creditLine', 'year', 'acquisitionYear', 'dimensions', 'width',
       'height', 'depth', 'units', 'inscription', 'thumbnailCopyright',
       'thumbnailUrl', 'url', 'birthYear', 'deathYear', 'year_numeric',
       'year_certainty', 'date_display', 'title_cleaned', 'has_inscription',
       'medium_group'],
      dtype='object')

In [64]:
# Drop columns
cols_to_drop = [
    'id',               # Redundant (using accession_number)
    'artistRole',       # Imbalanced, non-informative
    'width',            # Displayed in dimensions
    'height',           # Displayed in dimensions
    'depth',            # High missing values
    'units',            # No variance, information alreadt in 'dimensions'
    'inscription',      # Replaced by 'has_inscription'
    'birthYear',        # Already used for computation, redundant
    'deathYear',        # Already used for computation, redundant
    'year',              # Replaced by year_numeric
    'birthYear_artist_numeric',
    'deathYear_artist_numeric',
    'title',             # Replaced by 'title_cleaned'
    'dateText'
]




In [65]:
# Rename
df = df.drop(columns=cols_to_drop, errors='ignore')
df = df.rename(columns={'year_numeric': 'year', 'title_cleaned': 'title'})

In [66]:
df.columns

Index(['artist', 'artistId', 'medium', 'creditLine', 'acquisitionYear',
       'dimensions', 'thumbnailCopyright', 'thumbnailUrl', 'url', 'year',
       'year_certainty', 'date_display', 'title', 'has_inscription',
       'medium_group'],
      dtype='object')

In [67]:
# Re-order columns
df = df[['artist', 'artistId','title', 'medium', 'medium_group', 'creditLine', 'year','year_certainty',
         'acquisitionYear', 'dimensions', 'date_display', 'has_inscription', 'thumbnailCopyright', 'thumbnailUrl', 'url']]

In [68]:
df.columns

Index(['artist', 'artistId', 'title', 'medium', 'medium_group', 'creditLine',
       'year', 'year_certainty', 'acquisitionYear', 'dimensions',
       'date_display', 'has_inscription', 'thumbnailCopyright', 'thumbnailUrl',
       'url'],
      dtype='object')

In [69]:
df.dtypes

,0
artist,object
artistId,int64
title,object
medium,object
medium_group,object
creditLine,object
year,float64
year_certainty,object
acquisitionYear,float64
dimensions,object


In [70]:
df

,artist,artistId,title,medium,medium_group,creditLine,year,year_certainty,acquisitionYear,dimensions,date_display,has_inscription,thumbnailCopyright,thumbnailUrl,url
accession_number,,,,,,,,,,,,,,,
A00001,"Blake, Robert",38,A Figure Bowing before a Seated Old Man with h...,"Watercolour, ink, chalk and graphite on paper....",Work on Paper,Presented by Mrs John Richmond 1922,1784.0,Estimated,1922.0,419.0 x 394.0 mm,c. 1782–1787 (Artist Era),False,No copyright information available,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-a-fi...
A00002,"Blake, Robert",38,"Two Drawings of Frightened Figures, Probably f...",Graphite on paper,Work on Paper,Presented by Mrs John Richmond 1922,1784.0,Estimated,1922.0,213.0 x 311.0 mm,c. 1782–1787 (Artist Era),False,No copyright information available,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-two-...
A00003,"Blake, Robert",38,The Preaching of Warning. Verso: An Old Man En...,Graphite on paper. Verso: graphite on paper,Work on Paper,Presented by Mrs John Richmond 1922,1785.0,Verified,1922.0,467.0 x 343.0 mm,1785,False,No copyright information available,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...
A00004,"Blake, Robert",38,Six Drawings of Figures with Outstretched Arms,Graphite on paper,Work on Paper,Presented by Mrs John Richmond 1922,1784.0,Estimated,1922.0,394.0 x 318.0 mm,c. 1782–1787 (Artist Era),False,No copyright information available,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-six-...
A00005,"Blake, William",39,The Circle of the Lustful: Francesca da Rimini...,Line engraving on paper,Work on Paper,Purchased with the assistance of a special gra...,1826.0,Verified,1919.0,335.0 x 243.0 mm,1826,False,No copyright information available,http://www.tate.org.uk/art/images/work/A/A00/A...,http://www.tate.org.uk/art/artworks/blake-the-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T13865,"P-Orridge, Genesis",16646,Larvae (from Tampax Romana),"Perspex, Wood, hairpiece, tampon and human blood",Sculpture,Transferred from Tate Archive 2012,1975.0,Verified,2013.0,305.0 x 305.0 x 135.0 mm,1975,False,No copyright information available,No thumbnail URL available,http://www.tate.org.uk/art/artworks/p-orridge-...
T13866,"P-Orridge, Genesis",16646,Living Womb (from Tampax Romana),"Wood, Perspex, plastic, photograph on paper, t...",Modern Media,Transferred from Tate Archive 2012,1976.0,Verified,2013.0,305.0 x 305.0 x 135.0 mm,1976,False,No copyright information available,No thumbnail URL available,http://www.tate.org.uk/art/artworks/p-orridge-...
T13867,"Hatoum, Mona",2365,Present Tense,Soap and glass beads,Sculpture,Presented by Tate Members 2013,1996.0,Verified,2013.0,2410.0 x 45.0 x 2990.0 mm,1996,False,No copyright information available,No thumbnail URL available,http://www.tate.org.uk/art/artworks/hatoum-pre...


In [41]:
profile_cleaned = ProfileReport(df, title = "ARTWORKS_CLEAN_REPORT", config = config)

In [42]:
profile_cleaned

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 15/15 [00:04<00:00,  3.28it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [43]:
from google.colab import files

# 1. Save the dataframe to a CSV file in the Colab environment
# We use index=True to keep the 'accession_number' which is currently your index
df.to_csv('tate_artwork_cleaned.csv', index=True)

print("File saved to Colab. Starting download...")

# 2. Trigger the browser to download the file to your computer
files.download('tate_artwork_cleaned.csv')

File saved to Colab. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>